# Rovision — Subsea Defect Segmentation & Tracking Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshinnovaite/rovision/blob/main/rovision/notebooks/subsea_defect_demo.ipynb)

**Manual-prompt video tracking.** You mark a defect (a point or a box) on one frame; SAM 2 segments it and propagates the mask through the whole clip. Per frame we export a **mask + bounding box + pixel area + confidence** — the structured handoff point for a downstream classifier / severity / work-order layer (out of scope here).

> **What SAM 2 does and does not do.** It is a *promptable, class-agnostic* segmenter. It has no concept of "defect", "corrosion", or any category — it segments whatever you point it at and tracks it. Deciding *where* to prompt and *what the region means* is a separate stage. This demo isolates SAM 2's contribution: turning a prompt into a pixel-accurate, tracked mask plus its geometry.

### How to run
1. **Runtime → Change runtime type → GPU** (T4 is fine).
2. Run the cells top to bottom. Cell 1 installs SAM 2 and downloads the checkpoint.
3. At the **Upload** cell, drop in your video.
4. At the **Prompt** cell, read coordinates off the displayed frame grid and set `PROMPTS`.
5. Propagate, inspect the overlay video, and download the JSON/CSV of boxes.

## 1. Install SAM 2 + download checkpoint
Clones the repo, installs it, and pulls the **large** SAM 2.1 checkpoint (best accuracy — we're analysing offline, not in real time).

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

def find_repo_root(start=None):
    """Walk up from the working dir to find the sam2 repo root (has setup.py + sam2/)."""
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'setup.py').exists() and (cand / 'sam2').is_dir():
            return cand
    return None

REPO_ROOT = find_repo_root()

if REPO_ROOT is not None:
    # Running INSIDE the repo (e.g. VS Code Colab on a runtime that sees the repo):
    # install it in place — no clone needed.
    subprocess.run(f'cd "{REPO_ROOT}" && SAM2_BUILD_CUDA=0 pip install -e . && '
                   f'pip install huggingface_hub', shell=True, check=True)
elif IN_COLAB:
    # Not in the repo (vanilla remote Colab): clone + install.
    # Clone the rovision repo (which vendors SAM 2) and install it for `import sam2`.
    if not os.path.isdir('rovision_repo'):
        subprocess.run(['git', 'clone', '--quiet',
                        'https://github.com/joshinnovaite/rovision.git', 'rovision_repo'], check=True)
    subprocess.run('cd rovision_repo && SAM2_BUILD_CUDA=0 pip install -e . && '
                   'pip install huggingface_hub', shell=True, check=True)
    REPO_ROOT = Path('rovision_repo').resolve()
else:
    REPO_ROOT = Path('.').resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
importlib.invalidate_caches()

# REAL verification: raises if sam2 genuinely is not importable (no false "installed").
import sam2
print('SAM 2 import OK:', sam2.__file__)

# --- DOMAIN SWITCH -------------------------------------------------------------
# Which domain to process. 'subsea' reproduces the original demo exactly (empty
# suffix -> the original Drive folders/paths); 'pylon' (or any future domain)
# reads its taxonomy + uses its own per-domain paths so the two never collide.
DOMAIN = 'insulators'
DOM_SUFFIX = '' if DOMAIN == 'subsea' else f'_{DOMAIN}'

# Class list comes from the canonical registry rovision/domains.json (the single
# source of truth shared with the backend/frontend/labeller). Read it from the repo
# when present (local runs); on a Colab runtime that cloned an OLDER repo without the
# registry, fall back to the copy staged on Drive alongside the footage/labels.
import json as _json
def _load_registry():
    for c in [REPO_ROOT / 'rovision' / 'domains.json', Path('rovision/domains.json')]:
        if c.exists():
            return c, _json.load(open(c))
    if IN_COLAB:                      # repo clone is stale -> use the Drive-staged copy
        try:
            from google.colab import drive; drive.mount('/content/drive')
        except Exception as e:
            print('drive mount failed:', e)
        d = Path('/content/drive/MyDrive/rovision/domains.json')
        if d.exists():
            return d, _json.load(open(d))
    raise FileNotFoundError(
        "domains.json not found. The cloned repo predates the multi-domain registry; "
        "copy rovision/domains.json to MyDrive/rovision/domains.json (next to your "
        "labels file) and re-run this cell.")
_reg_path, DOMAINS = _load_registry()
assert DOMAIN in DOMAINS, f"unknown DOMAIN {DOMAIN!r}; known: {list(DOMAINS)}"
CLASSES = DOMAINS[DOMAIN]['all_classes']
print(f'DOMAIN = {DOMAIN}  |  registry: {_reg_path}  |  {len(CLASSES)} classes: {CLASSES}')

# Repo-relative data paths used by later cells (resolved from the repo root above).
FOOTAGE_DIR = REPO_ROOT / f'test_footage{DOM_SUFFIX}'
LABELS_PATH = REPO_ROOT / 'rovision' / 'labelling' / f'labels{DOM_SUFFIX}.json'
print('Repo root  :', REPO_ROOT)
print('Footage dir:', FOOTAGE_DIR, '| exists:', FOOTAGE_DIR.is_dir())
print('Labels file:', LABELS_PATH, '| exists:', LABELS_PATH.exists())

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. A LOCAL runtime has no NVIDIA GPU -> SAM 2 will be slow; '
          'use a Colab GPU runtime for speed.')


## 2. Imports, device & helpers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import json, csv, glob, subprocess

# Pick the best available device.
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Using device:', device)

# bfloat16 autocast on CUDA gives a big speed/memory win; TF32 on Ampere+.
if device.type == 'cuda':
    torch.autocast('cuda', dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

# ---- visualisation helpers (adapted from the official SAM 2 notebooks) ----
def show_mask(mask, ax, obj_id=None, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), [0.6]])
    else:
        cmap = plt.get_cmap('tab10')
        color = np.array([*cmap((0 if obj_id is None else obj_id) % 10)[:3], 0.6])
    h, w = mask.shape[-2:]
    ax.imshow(mask.reshape(h, w, 1) * color.reshape(1, 1, -1))

def show_points(coords, labels, ax, marker_size=200):
    pos = coords[labels == 1]; neg = coords[labels == 0]
    ax.scatter(pos[:, 0], pos[:, 1], color='lime', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg[:, 0], neg[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    ax.add_patch(plt.Rectangle((x0, y0), box[2]-x0, box[3]-y0, edgecolor='yellow', facecolor=(0,0,0,0), lw=2))

def mask_to_bbox_xywh(mask):
    """Tight bounding box [x, y, w, h] from a boolean mask, or None if empty."""
    ys, xs = np.where(mask)
    if xs.size == 0:
        return None
    x0, y0, x1, y1 = xs.min(), ys.min(), xs.max(), ys.max()
    return [int(x0), int(y0), int(x1 - x0 + 1), int(y1 - y0 + 1)]

## 3. Upload your video
Drop in your clip (`.mp4`, `.mov`, etc.). If you skip the upload, the demo falls back to the repo's bundled sample.

In [ ]:
VIDEO_PATH = None
if IN_COLAB:
    from google.colab import files
    print('Select your video file to upload...')
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = list(uploaded.keys())[0]

# Fallbacks if nothing was uploaded.
if VIDEO_PATH is None:
    for cand in ['test_footage/internal_assets.mov (720p).mp4']:
        if os.path.exists(cand):
            VIDEO_PATH = cand
            break

assert VIDEO_PATH and os.path.exists(VIDEO_PATH), 'No video found - upload one above.'
print('Using video:', VIDEO_PATH)


## 4. Extract frames
SAM 2's video predictor consumes a **directory of JPEG frames** named `00000.jpg, 00001.jpg, ...`, not an `.mp4` directly. We use `ffmpeg` to split the clip.

**`FRAME_STRIDE`** — for slow-moving ROV footage you rarely need every frame. A stride of 3 keeps every 3rd frame, cutting memory/runtime ~3x. Increase it for long clips; set to 1 for full temporal detail.

> **Resolution note.** SAM 2 resizes every frame to 1024x1024 internally, so feeding 4K doesn't add model detail and hairline defects can be lost at that scale. 720p is already near the sweet spot. If small defects get missed, the fix is *tiling* (run on crops) — a later step, not needed for this first look.

In [ ]:
FRAME_STRIDE = 3          # keep every Nth frame (1 = all frames)
MAX_FRAMES = 200          # safety cap so a long clip can't exhaust memory
FRAMES_DIR = 'frames'

# Probe the source video.
def ffprobe(path):
    try:
        out = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0',
            '-show_entries', 'stream=width,height,r_frame_rate', '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1', path], capture_output=True, text=True)
        return out.stdout
    except FileNotFoundError:
        return '(ffprobe not available)'
print(ffprobe(VIDEO_PATH))

# Extract frames. select='not(mod(n,STRIDE))' keeps every Nth source frame.
!rm -rf "$FRAMES_DIR" && mkdir -p "$FRAMES_DIR"
vf = f"select=not(mod(n\\,{FRAME_STRIDE}))"
subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-i', VIDEO_PATH,
                '-vf', vf, '-vsync', '0', '-q:v', '2', '-frames:v', str(MAX_FRAMES),
                f'{FRAMES_DIR}/%05d.jpg'], check=True)

frame_names = sorted(glob.glob(f'{FRAMES_DIR}/*.jpg'))
assert frame_names, 'No frames extracted — is ffmpeg installed and the video readable?'
H, W = np.array(Image.open(frame_names[0])).shape[:2]
print(f'Extracted {len(frame_names)} frames at {W}x{H} (stride={FRAME_STRIDE}).')

## 5. Load the SAM 2 video predictor
We load the **large SAM 2.1** model from Hugging Face (no manual checkpoint juggling). The offload flags keep memory low so long clips survive on a modest GPU.

In [ ]:
from sam2.sam2_video_predictor import SAM2VideoPredictor

predictor = SAM2VideoPredictor.from_pretrained('facebook/sam2.1-hiera-large', device=device)

# offload_*_to_cpu trade a little speed for much lower GPU memory — important for long clips.
state = predictor.init_state(video_path=FRAMES_DIR,
                             offload_video_to_cpu=True,
                             offload_state_to_cpu=True)
print('Inference state initialised over', len(frame_names), 'frames.')

## 6. Choose a prompt frame and read off coordinates
The frame below is shown with a pixel grid. Find the defect you want to track and note its (x, y) pixel location — or a bounding box — from the axes.

In [ ]:
PROMPT_FRAME_IDX = 0   # which extracted frame to place the prompt on

img = np.array(Image.open(frame_names[PROMPT_FRAME_IDX]))
fig, ax = plt.subplots(figsize=(12, 12 * H / W))
ax.imshow(img)
ax.set_title(f'Frame {PROMPT_FRAME_IDX} — read off defect coordinates')
ax.set_xticks(np.arange(0, W, max(1, W // 20)))
ax.set_yticks(np.arange(0, H, max(1, H // 20)))
ax.grid(color='cyan', alpha=0.3, linewidth=0.5)
plt.show()

## 7. Define and apply the prompt
Set `PROMPTS` below. You can use **points** (with `1` = positive / part of the defect, `0` = negative / exclude background) and/or a **box**. Multiple positive + negative clicks usually give the cleanest mask.

Each entry is one **object** to track. Add more dicts to track multiple defects at once.

In [ ]:
# --- EDIT THESE to match your footage (read coords off the grid above) ---
PROMPTS = [
    {
        'obj_id': 1,
        'points': [[W // 2, H // 2]],   # [[x, y], ...]  positive click(s) on the defect
        'labels': [1],                  # 1 = positive, 0 = negative (exclude)
        'box': None,                    # or [x0, y0, x1, y1]
    },
]
MULTIMASK = True   # return 3 granularities and auto-pick the best by confidence; good for ambiguous prompts

predictor.reset_state(state)
fig, ax = plt.subplots(figsize=(12, 12 * H / W))
ax.imshow(img); ax.set_title(f'Prompt result on frame {PROMPT_FRAME_IDX}')

for p in PROMPTS:
    pts = np.array(p['points'], dtype=np.float32) if p['points'] else None
    lbls = np.array(p['labels'], dtype=np.int32) if p['points'] else None
    box = np.array(p['box'], dtype=np.float32) if p['box'] else None
    _, obj_ids, mask_logits = predictor.add_new_points_or_box(
        inference_state=state, frame_idx=PROMPT_FRAME_IDX, obj_id=p['obj_id'],
        points=pts, labels=lbls, box=box)
    # Visualise.
    for i, oid in enumerate(obj_ids):
        show_mask((mask_logits[i] > 0.0).cpu().numpy(), ax, obj_id=oid)
    if pts is not None:
        show_points(pts, lbls, ax)
    if box is not None:
        show_box(box, ax)
plt.show()
print('If the mask is poor: add negative clicks (label 0) on background, add more positive clicks, or try a box.')

## 8. Propagate through the whole clip
Tracks every prompted object across all frames and collects the masks.

In [ ]:
video_segments = {}  # frame_idx -> { obj_id -> boolean mask }
for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(state):
    video_segments[frame_idx] = {
        oid: (mask_logits[i] > 0.0).cpu().numpy().squeeze()
        for i, oid in enumerate(obj_ids)
    }
print('Propagated across', len(video_segments), 'frames.')

## 9. Structured output: boxes, areas, confidence → JSON + CSV
This is the **handoff to a downstream layer**. Per frame, per object we record the tight bounding box (XYWH), pixel area, and mask-coverage fraction — the geometry a severity/classifier stage would consume.

In [ ]:
records = []
for f_idx in sorted(video_segments):
    for oid, mask in video_segments[f_idx].items():
        bbox = mask_to_bbox_xywh(mask)
        if bbox is None:
            continue  # object not visible this frame
        area = int(mask.sum())
        records.append({
            'frame_idx': int(f_idx),
            'obj_id': int(oid),
            'bbox_xywh': bbox,
            'area_px': area,
            'coverage_frac': round(area / (H * W), 5),
        })

with open('defect_tracks.json', 'w') as fh:
    json.dump(records, fh, indent=2)
with open('defect_tracks.csv', 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['frame_idx', 'obj_id', 'x', 'y', 'w', 'h', 'area_px', 'coverage_frac'])
    for r in records:
        w.writerow([r['frame_idx'], r['obj_id'], *r['bbox_xywh'], r['area_px'], r['coverage_frac']])

print(f'Wrote {len(records)} rows to defect_tracks.json / defect_tracks.csv')
records[:5]

## 10. Render an overlay video to eyeball quality
Burns the tracked masks + boxes onto the frames and stitches them back into an `.mp4`.

In [ ]:
OUT_DIR = 'overlay'; OUT_VIDEO = 'defect_overlay.mp4'
!rm -rf "$OUT_DIR" && mkdir -p "$OUT_DIR"

for i, fname in enumerate(frame_names):
    fig, ax = plt.subplots(figsize=(W / 100, H / 100), dpi=100)
    ax.imshow(np.array(Image.open(fname))); ax.axis('off')
    for oid, mask in video_segments.get(i, {}).items():
        show_mask(mask, ax, obj_id=oid)
        bbox = mask_to_bbox_xywh(mask)
        if bbox:
            x, y, bw, bh = bbox
            ax.add_patch(plt.Rectangle((x, y), bw, bh, edgecolor='yellow', facecolor='none', lw=1.5))
            ax.text(x, max(0, y - 5), f'obj {oid}', color='yellow', fontsize=9, weight='bold')
    ax.set_position([0, 0, 1, 1])
    fig.savefig(f'{OUT_DIR}/{i:05d}.jpg', bbox_inches='tight', pad_inches=0)
    plt.close(fig)

# ~8 fps playback so slow footage is watchable; adjust -r to taste.
subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-framerate', '8',
                '-i', f'{OUT_DIR}/%05d.jpg', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', OUT_VIDEO], check=True)
print('Wrote', OUT_VIDEO)

In [ ]:
# Preview inline (Colab).
if IN_COLAB:
    from IPython.display import HTML
    from base64 import b64encode
    data = b64encode(open(OUT_VIDEO, 'rb').read()).decode()
    display(HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

## 11. Download the results

In [ ]:
if IN_COLAB:
    from google.colab import files
    for f in ['defect_overlay.mp4', 'defect_tracks.json', 'defect_tracks.csv']:
        if os.path.exists(f):
            files.download(f)

## 12. Verify the golden-set prompts

Load `rovision/labelling/labels.json` and, for the **currently-uploaded video**, run SAM 2 (image mode) using your hand-drawn **boxes** as prompts on the exact frames you labelled. This checks each prompt produces a sensible mask *before* we trust the golden set for propagation/training.

> Self-contained: needs cells 1-2 (install/imports) + a video uploaded at cell 3 + `labels.json`. Verify **one video at a time** — upload that video, run these cells, repeat. (You can skip the single-prompt cells 4-11.)

In [ ]:
# Load + group the golden-set labels.
import json, os, cv2
from collections import defaultdict

# LABELS_PATH was set (repo-relative) in the install cell; fall back to upload.
if not (isinstance(LABELS_PATH, (str, os.PathLike)) and os.path.exists(LABELS_PATH)):
    if IN_COLAB:
        from google.colab import files
        print('labels.json not found in the repo - upload rovision/labelling/labels.json ...')
        LABELS_PATH = next(iter(files.upload()))
    else:
        raise FileNotFoundError('labels.json not found')

labels = json.load(open(LABELS_PATH))
by_video = defaultdict(lambda: defaultdict(list))
for r in labels:
    by_video[r['video']][int(r['frame'])].append(r)
print(f"{len(labels)} labels across {len(by_video)} videos:")
for v in by_video:
    print(f"  {sum(len(x) for x in by_video[v].values()):3d} boxes  {v}")


In [ ]:
# Load SAM 2 in IMAGE mode (separate from the video predictor) + class colours.
from sam2.sam2_image_predictor import SAM2ImagePredictor
img_predictor = SAM2ImagePredictor.from_pretrained('facebook/sam2.1-hiera-large', device=device)

CLASS_LIST = ['corrosion','coating_breakdown','surface_deposit','marine_growth',
              'trash_rack_blockage','seal_joint_degradation','pipework','ladder',
              'outlet_inlet','valve_mixer','fish','coral','seagrass',
              'rov_manipulator','dropped_object']

def class_color(cls):
    i = CLASS_LIST.index(cls) if cls in CLASS_LIST else 0
    return np.array(plt.get_cmap('tab20')(i % 20)[:3])

In [ ]:
# Verify the prompts for the uploaded video: box -> SAM 2 mask, overlaid per frame.
vname = os.path.basename(VIDEO_PATH)
cands = [k for k in by_video if k == vname or os.path.splitext(k)[0] in vname]
assert cands, f"No labels match uploaded video '{vname}'. Upload the matching video at cell 3."
VERIFY_VIDEO = cands[0]
frames_with_labels = sorted(by_video[VERIFY_VIDEO])
print(f"Verifying {VERIFY_VIDEO}: {len(frames_with_labels)} labelled frames")

cap = cv2.VideoCapture(VIDEO_PATH)
for fidx in frames_with_labels:
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ok, bgr = cap.read()
    if not ok:
        print(f"  ! could not read frame {fidx}"); continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    recs = by_video[VERIFY_VIDEO][fidx]
    boxes = np.array([r['bbox'] for r in recs], dtype=np.float32)
    with torch.inference_mode():
        img_predictor.set_image(rgb)
        masks, scores, _ = img_predictor.predict(box=boxes, multimask_output=False)
    H, W = rgb.shape[:2]
    masks = np.asarray(masks).reshape(len(recs), H, W)
    scores = np.atleast_1d(np.asarray(scores).ravel())

    overlay = np.zeros((H, W, 4))
    fig, ax = plt.subplots(figsize=(13, 13 * H / W))
    ax.imshow(rgb); ax.axis('off')
    ax.set_title(f"{VERIFY_VIDEO}  frame {fidx}  ({len(recs)} prompts)", fontsize=10)
    for r, m, sc in zip(recs, masks, scores):
        col = class_color(r['defect'])
        overlay[m > 0.0] = [*col, 0.5]
        x0, y0, x1, y1 = r['bbox']
        ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, ec=col, fc='none', lw=2))
        ax.text(x0, max(0, y0-5), f"{r['defect']} {sc:.2f}", color='white',
                fontsize=8, weight='bold',
                bbox=dict(facecolor=col, alpha=0.7, pad=1, edgecolor='none'))
    ax.imshow(overlay)
    plt.show()
cap.release()

## 13. Propagate prompts -> amplified detection dataset

For the currently-uploaded video, SAM 2 propagates each golden-set **box** forward across a short window of frames, turning every seed box into ~`WINDOW_N` labelled frames. This manufactures the detector training set. Outputs:
- `dataset/images/<video>_<frame>.jpg` -- the frames
- `dataset/amplified_labels.json` -- boxes (xyxy) + class per frame

Run **once per video**: upload the video at section 3, run the section-12 "load labels" cell (defines `by_video`), then run this. It accumulates across runs; re-running a video replaces that video's records.

> Heavy step -- expect minutes per video on a T4. Deliberately overfit-friendly: propagation stays close to your verified seeds. (Needs section 1-2 + section 3 + the section-12 load-labels cell; the image predictor / verify cells are not required here.)

In [ ]:
import os, json, shutil
import numpy as np
from sam2.sam2_video_predictor import SAM2VideoPredictor

# Reuse a video predictor if one is already loaded, else create it.
try:
    vpredictor
except NameError:
    vpredictor = SAM2VideoPredictor.from_pretrained('facebook/sam2.1-hiera-large', device=device)

# --- config ---
WINDOW_N = 25        # labelled frames produced per seed (forward from the seed)
WINDOW_STRIDE = 3    # original-frame step within a window (slow ROV footage -> small motion)
DATASET_DIR = 'dataset'
IMG_DIR = os.path.join(DATASET_DIR, 'images')
TMP = 'prop_tmp'
os.makedirs(IMG_DIR, exist_ok=True)

def mask_to_xyxy(m):
    ys, xs = np.where(m)
    if xs.size == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]

# Match the uploaded video to a labels entry (one video per run, like section 12).
vname = os.path.basename(VIDEO_PATH)
cands = [k for k in by_video if k == vname or os.path.splitext(k)[0] in vname]
assert cands, f"No labels match uploaded video '{vname}'. Upload the matching video at section 3."
VID = cands[0]
stem = os.path.splitext(VID)[0]

cap = cv2.VideoCapture(VIDEO_PATH)
T = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
print(f"Propagating {VID}: {len(by_video[VID])} seed frames, T={T}")

amp_records = []
for si, seed_frame in enumerate(sorted(by_video[VID])):
    recs = by_video[VID][seed_frame]
    idxs = [seed_frame + k*WINDOW_STRIDE for k in range(WINDOW_N)]
    idxs = [i for i in idxs if 0 <= i < T]
    if len(idxs) < 2:
        continue
    # extract the window to TMP as sequential jpgs
    if os.path.isdir(TMP): shutil.rmtree(TMP)
    os.makedirs(TMP)
    pos_to_orig = {}
    for pos, oi in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, oi)
        ok, bgr = cap.read()
        if not ok:
            break
        cv2.imwrite(os.path.join(TMP, f"{pos:05d}.jpg"), bgr)
        pos_to_orig[pos] = oi
    if len(pos_to_orig) < 2:
        continue

    state = vpredictor.init_state(video_path=TMP, offload_video_to_cpu=True,
                                  offload_state_to_cpu=True)
    objid_to_cls = {}
    for j, r in enumerate(recs):
        box = np.array(r['bbox'], dtype=np.float32)
        with torch.inference_mode():
            vpredictor.add_new_points_or_box(inference_state=state, frame_idx=0,
                                             obj_id=j, box=box)
        objid_to_cls[j] = r['defect']

    with torch.inference_mode():
        for pos, obj_ids, mask_logits in vpredictor.propagate_in_video(state):
            orig = pos_to_orig.get(pos)
            if orig is None:
                continue
            img_name = f"{stem}_{orig:06d}.jpg"
            img_path = os.path.join(IMG_DIR, img_name)
            if not os.path.exists(img_path):
                shutil.copy(os.path.join(TMP, f"{pos:05d}.jpg"), img_path)
            for k, oid in enumerate(obj_ids):
                m = (mask_logits[k] > 0.0).cpu().numpy().squeeze()
                bb = mask_to_xyxy(m)
                if bb is None:
                    continue
                amp_records.append({'video': VID, 'image': img_name, 'frame': int(orig),
                                    'defect': objid_to_cls[oid], 'bbox': bb})
    vpredictor.reset_state(state)
    print(f"  seed {si+1}/{len(by_video[VID])} (frame {seed_frame}) done")

cap.release()
if os.path.isdir(TMP): shutil.rmtree(TMP)

# accumulate into the dataset json (replace this video's prior records)
AMP_PATH = os.path.join(DATASET_DIR, 'amplified_labels.json')
existing = json.load(open(AMP_PATH)) if os.path.exists(AMP_PATH) else []
existing = [r for r in existing if r['video'] != VID] + amp_records
json.dump(existing, open(AMP_PATH, 'w'), indent=1)
n_frames = len({r['image'] for r in amp_records})
print(f"\n{VID}: +{len(amp_records)} boxes across {n_frames} frames. "
      f"Dataset total: {len(existing)} boxes -> {AMP_PATH}")

## 14. Batch amplification over ALL videos (one cell, resumable, via Google Drive)

This replaces the per-video hand-holding. **One-time setup in Drive:**
```
MyDrive/rovision/
  test_footage/   <- put ALL your videos here
  labels.json     <- your golden set (rovision/labelling/labels.json)
```
Then just run this cell. It mounts Drive, loops every labelled video, and writes the dataset (`images/` + `amplified_labels.json`) **back to Drive** after each video -- so it survives the runtime expiring. Re-run after a disconnect and it **skips videos already done** (`RESUME = True`). Needs only sections 1-2 first.

In [ ]:
import os, json, shutil, glob, cv2
import numpy as np
from sam2.sam2_video_predictor import SAM2VideoPredictor

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# --- EDIT these to point at YOUR Drive folders ---
DRIVE_ROOT      = '/content/drive/MyDrive/rovision'
_SUF            = '' if DOMAIN == 'subsea' else f'_{DOMAIN}'   # DOMAIN set in the install cell
DRIVE_VIDEO_DIR = os.path.join(DRIVE_ROOT, f'test_footage{_SUF}')
LABELS_FILE     = os.path.join(DRIVE_ROOT, f'labels{_SUF}.json')
DRIVE_DATASET   = os.path.join(DRIVE_ROOT, f'dataset{_SUF}')   # output (persists on Drive)
RESUME = True            # skip videos already present in the output json
WINDOW_N, WINDOW_STRIDE = 25, 3

IMG_DIR  = os.path.join(DRIVE_DATASET, 'images')
AMP_PATH = os.path.join(DRIVE_DATASET, 'amplified_labels.json')
TMP = 'prop_tmp'
os.makedirs(IMG_DIR, exist_ok=True)

try:
    vpredictor
except NameError:
    vpredictor = SAM2VideoPredictor.from_pretrained('facebook/sam2.1-hiera-large', device=device)

labels = json.load(open(LABELS_FILE))
by_video = {}
for r in labels:
    by_video.setdefault(r['video'], {}).setdefault(int(r['frame']), []).append(r)

def mask_to_xyxy(m):
    ys, xs = np.where(m)
    return None if xs.size == 0 else [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]

def amplify(video_path, VID):
    stem = os.path.splitext(VID)[0]
    cap = cv2.VideoCapture(video_path)
    T = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    out = []
    for seed_frame in sorted(by_video[VID]):
        recs = by_video[VID][seed_frame]
        idxs = [i for i in (seed_frame + k*WINDOW_STRIDE for k in range(WINDOW_N)) if 0 <= i < T]
        if len(idxs) < 2:
            continue
        if os.path.isdir(TMP): shutil.rmtree(TMP)
        os.makedirs(TMP)
        pos_to_orig = {}
        for pos, oi in enumerate(idxs):
            cap.set(cv2.CAP_PROP_POS_FRAMES, oi); ok, bgr = cap.read()
            if not ok: break
            cv2.imwrite(os.path.join(TMP, f"{pos:05d}.jpg"), bgr); pos_to_orig[pos] = oi
        if len(pos_to_orig) < 2:
            continue
        state = vpredictor.init_state(video_path=TMP, offload_video_to_cpu=True,
                                      offload_state_to_cpu=True)
        objid_to_cls = {}
        for j, r in enumerate(recs):
            with torch.inference_mode():
                vpredictor.add_new_points_or_box(inference_state=state, frame_idx=0, obj_id=j,
                                                 box=np.array(r['bbox'], dtype=np.float32))
            objid_to_cls[j] = r['defect']
        with torch.inference_mode():
            for pos, obj_ids, mask_logits in vpredictor.propagate_in_video(state):
                orig = pos_to_orig.get(pos)
                if orig is None: continue
                img_name = f"{stem}_{orig:06d}.jpg"
                ip = os.path.join(IMG_DIR, img_name)
                if not os.path.exists(ip): shutil.copy(os.path.join(TMP, f"{pos:05d}.jpg"), ip)
                for k, oid in enumerate(obj_ids):
                    m = (mask_logits[k] > 0.0).cpu().numpy().squeeze()
                    bb = mask_to_xyxy(m)
                    if bb is None: continue
                    out.append({'video': VID, 'image': img_name, 'frame': int(orig),
                                'defect': objid_to_cls[oid], 'bbox': bb})
        vpredictor.reset_state(state)
    cap.release()
    return out

drive_videos = {os.path.basename(p): p for p in glob.glob(os.path.join(DRIVE_VIDEO_DIR, '*'))
                if os.path.splitext(p)[1].lower() in {'.mp4', '.mov', '.avi', '.mkv'}}
all_records = json.load(open(AMP_PATH)) if os.path.exists(AMP_PATH) else []
done = {r['video'] for r in all_records}

for VID in by_video:
    match = drive_videos.get(VID) or next((p for n, p in drive_videos.items()
                                           if os.path.splitext(VID)[0] in n), None)
    if match is None:
        print(f"SKIP {VID}: no matching file in {DRIVE_VIDEO_DIR}"); continue
    if RESUME and VID in done:
        print(f"skip {VID}: already done"); continue
    print(f"-> {VID} ...", flush=True)
    recs = amplify(match, VID)
    all_records = [r for r in all_records if r['video'] != VID] + recs
    json.dump(all_records, open(AMP_PATH, 'w'), indent=1)   # save to Drive after EACH video
    done.add(VID)
    print(f"   +{len(recs)} boxes; total {len(all_records)} (saved to Drive)")

if os.path.isdir(TMP): shutil.rmtree(TMP)
print(f"\nDONE. {len(all_records)} boxes / {len({r['image'] for r in all_records})} frames in {DRIVE_DATASET}")

## 15. Convert amplified dataset -> YOLO format

Reads `dataset/amplified_labels.json` + images from Drive and writes a YOLO dataset to the **local runtime disk** (`/content/yolo_dataset`, fast training I/O): `images/{train,val}`, `labels/{train,val}`, and `data.yaml` with the 15 classes. Boxes are converted xyxy -> normalized `cx cy w h`.

In [ ]:
import os, json, shutil, cv2

# --- config ---
DRIVE_ROOT  = '/content/drive/MyDrive/rovision'
_SUF        = '' if DOMAIN == 'subsea' else f'_{DOMAIN}'   # DOMAIN set in the install cell
AMP_JSON    = os.path.join(DRIVE_ROOT, f'dataset{_SUF}', 'amplified_labels.json')
SRC_IMG_DIR = os.path.join(DRIVE_ROOT, f'dataset{_SUF}', 'images')
OUT_DIR     = '/content/yolo_dataset'     # local disk -> fast training
VAL_EVERY   = 7                           # 1-in-N images -> val (deterministic)
# CLASSES is set from the registry in the install cell; fall back to the subsea
# list if this cell is run standalone.
CLASSES = globals().get('CLASSES') or ['corrosion','coating_breakdown','surface_deposit',
           'marine_growth','trash_rack_blockage','seal_joint_degradation','pipework','ladder',
           'outlet_inlet','valve_mixer','fish','coral','seagrass','rov_manipulator','dropped_object']
cls_id = {c: i for i, c in enumerate(CLASSES)}

if os.path.isdir(OUT_DIR):
    shutil.rmtree(OUT_DIR)
for sp in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(OUT_DIR, sp), exist_ok=True)

recs = json.load(open(AMP_JSON))
by_img = {}
for r in recs:
    by_img.setdefault(r['image'], []).append(r)

n_box = {'train': 0, 'val': 0}; n_img = {'train': 0, 'val': 0}; missing = 0
for i, (img, rs) in enumerate(sorted(by_img.items())):
    src = os.path.join(SRC_IMG_DIR, img)
    im = cv2.imread(src) if os.path.exists(src) else None
    if im is None:
        missing += 1; continue
    H, W = im.shape[:2]
    split = 'val' if (i % VAL_EVERY == 0) else 'train'
    shutil.copy(src, os.path.join(OUT_DIR, f'images/{split}', img))
    lines = []
    for r in rs:
        if r['defect'] not in cls_id:
            continue
        x0, y0, x1, y1 = r['bbox']
        cx, cy = ((x0 + x1) / 2) / W, ((y0 + y1) / 2) / H
        bw, bh = (x1 - x0) / W, (y1 - y0) / H
        cx, cy, bw, bh = (min(1, max(0, v)) for v in (cx, cy, bw, bh))
        if bw <= 0 or bh <= 0:
            continue
        lines.append(f"{cls_id[r['defect']]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        n_box[split] += 1
    stem = os.path.splitext(img)[0]
    with open(os.path.join(OUT_DIR, f'labels/{split}', stem + '.txt'), 'w') as f:
        f.write('\n'.join(lines))
    n_img[split] += 1

yaml = f"path: {OUT_DIR}\ntrain: images/train\nval: images/val\nnames:\n"
yaml += ''.join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES))
with open(os.path.join(OUT_DIR, 'data.yaml'), 'w') as f:
    f.write(yaml)

print(f"train: {n_img['train']} imgs / {n_box['train']} boxes")
print(f"val:   {n_img['val']} imgs / {n_box['val']} boxes")
print(f"missing images skipped: {missing}")
print('data.yaml ->', os.path.join(OUT_DIR, 'data.yaml'))

## 16. Train YOLO11n (overfit-friendly)

COCO-pretrained `yolo11n`, no early stopping (we *want* to overfit the demo footage), weights saved back to Drive so they persist. Bump `imgsz` to 1280 if small defects are missed; drop `batch` if you hit OOM.

In [ ]:
import subprocess, os, shutil
subprocess.run('pip install -q ultralytics', shell=True, check=True)
from ultralytics import YOLO

_SUF = '' if DOMAIN == 'subsea' else f'_{DOMAIN}'
# Per-domain training recipe. Subsea defects are large, frame-filling blobs, so the
# original 960px + default augmentation overfits fine. Pylon parts are THIN and SMALL
# in aerial frames (cross_arm/pylon_top/insulator), so train near-native resolution
# and switch OFF the object-shrinking augmentations: mosaic tiles 4 images (~2x extra
# downscale + clips thin lattice) and scale jitter shrinks further. This is the fix
# for the "thin classes train to mAP 0" failure.
if DOMAIN == 'subsea':
    TRAIN_ARGS = dict(epochs=80, imgsz=960, batch=16)
else:
    # Use the PROVEN subsea recipe verbatim (the one config known to train a confident
    # model). Isolates recipe-vs-data: if this trains confidently, our earlier tweaks
    # (imgsz/batch/epochs) were the culprit; if it still collapses, the data is.
    TRAIN_ARGS = dict(epochs=80, imgsz=960, batch=16)

model = YOLO('yolo11n.pt')          # COCO-pretrained nano
# ROOT-CAUSE FIX: cell 4 (§2) enters a GLOBAL torch.autocast('cuda', bfloat16) for SAM 2
# speed and never exits it, so it is still active here. Training YOLO inside that leaked
# autocast casts the EMA/optimizer updates to bf16 and corrupts the EMA model that gets
# validated + saved as best.pt -> val mAP 0 and a dead best.pt. Shield training from it;
# YOLO manages its own AMP internally.
import torch
with torch.autocast('cuda', enabled=False):
    model.train(data='/content/yolo_dataset/data.yaml', device=0, patience=0,
                name=f'rovision_yolo11n{_SUF}', **TRAIN_ARGS)

best = f'runs/detect/rovision_yolo11n{_SUF}/weights/best.pt'
dst = f'/content/drive/MyDrive/rovision/rovision_yolo11n{_SUF}_best.pt'
if os.path.exists(best):
    shutil.copy(best, dst)
    print('saved best weights ->', dst)

## 17. Detect -> Segment demo (the capstone)

Runs the trained YOLO11n with **ByteTrack** so each defect is a *tracked instance* (stable ID, colour = instance, label = class) rather than an independent per-frame box -- this removes the per-frame flicker. SAM 2 refines to masks (+ area/coverage) every `REFINE_EVERY` frames; tracks shorter than `MIN_TRACK_LEN` frames are dropped as blips (two-pass: track -> filter -> render). The window defaults to a **seed region (~40s)** so it runs on footage the model actually overfit (per DR-005), not unseen frames. Outputs an overlay video, `detection_cards.json` (per-frame) and `track_cards.json` (per-instance -- the seed of the dashboard's cards rail). Needs sections 1-2; reads model + video from Drive.

In [ ]:
import os, json, shutil, subprocess
import numpy as np, cv2
from collections import defaultdict
from sam2.sam2_image_predictor import SAM2ImagePredictor
try:
    from ultralytics import YOLO
except ImportError:
    subprocess.run('pip install -q ultralytics', shell=True, check=True)
    from ultralytics import YOLO

if IN_COLAB and not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

# --- config ---
_SUF       = '' if DOMAIN == 'subsea' else f'_{DOMAIN}'   # DOMAIN set in the install cell
MODEL_PATH = f'/content/drive/MyDrive/rovision/rovision_yolo11n{_SUF}_best.pt'
OUT_DIR    = f'/content/drive/MyDrive/rovision/demo{_SUF}'
# Per-domain hero clip + the seed region the model overfit on. EDIT VIDEO_PATH to
# the clip you want as the bundle (it must live in the domain's Drive footage dir,
# and its filename must match exactly).
if DOMAIN == 'subsea':
    VIDEO_PATH = '/content/drive/MyDrive/rovision/test_footage/internal_assets.mov (720p).mp4'
    START_SEC, DURATION_SEC = 40, 15
    ASSET = 'internal_assets'
else:
    # hero clip for the insulators domain (lineman working on a tower; short single pan).
    VIDEO_PATH = f'/content/drive/MyDrive/rovision/test_footage{_SUF}/maw_3.mp4'
    START_SEC, DURATION_SEC = 0, 9   # maw clips are short single pans; ~whole clip. Set to your richest labelled clip.
    ASSET = 'generic'
# Detector inference at the training imgsz. This is a proper model now (worker is
# COCO-pretrained -> confident), so use a normal conf; tune after eyeballing the overlay.
CONF       = 0.5 if DOMAIN == 'subsea' else 0.25
DET_IMGSZ  = 960
REFINE_EVERY = 10                   # SAM 2 mask refinement every Nth frame
MIN_TRACK_LEN = 5 if DOMAIN == 'subsea' else 3
TRACKER = 'bytetrack.yaml'
os.makedirs(OUT_DIR, exist_ok=True)

_PAL = [(230,25,75),(60,180,75),(255,225,25),(0,130,200),(245,130,48),(145,30,180),
        (70,240,240),(240,50,230),(210,245,60),(250,190,212),(0,128,128),(220,190,255),
        (170,110,40),(190,255,180),(128,0,0),(255,215,180),(0,0,128),(128,128,128)]
def col_rgb(i): return _PAL[i % len(_PAL)]

# Mask -> normalized polygon contours (for the web app to render masks as a
# separate, selectable layer; tiny on disk vs bitmaps). Refine frames only.
POLY_EPS_FRAC      = 0.004    # approxPolyDP epsilon as a fraction of contour arc length
POLY_MIN_AREA_FRAC = 0.0008   # drop contour islands smaller than this fraction of the frame
POLY_MAX_CONTOURS  = 4        # keep the N largest contours per mask (anti-blowup)
def mask_to_polygons(mask_bool, W, H):
    m = (mask_bool.astype('uint8')) * 255
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    frame_area = float(W * H)
    scored = []
    for c in cnts:
        a = cv2.contourArea(c)
        if a < POLY_MIN_AREA_FRAC * frame_area:
            continue
        approx = cv2.approxPolyDP(c, POLY_EPS_FRAC * cv2.arcLength(c, True), True).reshape(-1, 2)
        if len(approx) < 3:
            continue
        scored.append((a, [[round(float(x)/W, 4), round(float(y)/H, 4)] for x, y in approx]))
    scored.sort(key=lambda t: -t[0])
    return [poly for _, poly in scored[:POLY_MAX_CONTOURS]]

det = YOLO(MODEL_PATH)
NAMES = det.names
refiner = SAM2ImagePredictor.from_pretrained('facebook/sam2.1-hiera-large', device=device)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
start_f, end_f = int(START_SEC*fps), int((START_SEC+DURATION_SEC)*fps)
print(f"fps={fps:.1f}; frames {start_f}-{end_f}; tracker={TRACKER}, conf={CONF}")

TMP = 'demo_frames'; shutil.rmtree(TMP, ignore_errors=True); os.makedirs(TMP)

# --- Pass 1: track + refine; buffer raw frames and per-frame detections ---
per_frame = []
track_len = defaultdict(int)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
fidx, k = start_f, 0
while fidx < end_f:
    ok, bgr = cap.read()
    if not ok:
        break
    cv2.imwrite(f"{TMP}/{k:05d}.jpg", bgr)
    H, W = bgr.shape[:2]
    res = det.track(bgr, persist=True, conf=CONF, tracker=TRACKER, imgsz=DET_IMGSZ, verbose=False)[0]
    dets = []
    if res.boxes is not None and res.boxes.id is not None:
        boxes = res.boxes.xyxy.cpu().numpy()
        clss  = res.boxes.cls.cpu().numpy().astype(int)
        confs = res.boxes.conf.cpu().numpy()
        ids   = res.boxes.id.cpu().numpy().astype(int)
        masks = None
        if (k % REFINE_EVERY == 0) and len(boxes):
            with torch.inference_mode():
                refiner.set_image(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
                m, _, _ = refiner.predict(box=boxes, multimask_output=False)
            masks = np.asarray(m).reshape(len(boxes), H, W) > 0.0
        for i in range(len(boxes)):
            d = {'id': int(ids[i]), 'cls': int(clss[i]), 'conf': float(confs[i]),
                 'bbox': [int(v) for v in boxes[i]]}
            if masks is not None:
                d['mask'] = masks[i]
            dets.append(d)
            track_len[int(ids[i])] += 1
    per_frame.append({'k': k, 'frame': int(fidx), 'dets': dets})
    fidx += 1; k += 1
cap.release()

keep = {tid for tid, n in track_len.items() if n >= MIN_TRACK_LEN}
print(f"{len(track_len)} raw tracks -> {len(keep)} kept (>= {MIN_TRACK_LEN} frames)")

# --- encode the RAW clip (no overlay) from the buffered Pass-1 frames, BEFORE
# Pass 2 overwrites them. +faststart is REQUIRED for browser range-seeking. ---
raw_mp4 = '/content/raw.mp4'
subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
                '-framerate', str(round(fps)), '-i', f'{TMP}/%05d.jpg',
                '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2',
                '-c:v', 'libx264', '-profile:v', 'baseline', '-level', '3.1',
                '-pix_fmt', 'yuv420p', '-movflags', '+faststart', raw_mp4], check=True)
shutil.copy(raw_mp4, os.path.join(OUT_DIR, 'raw.mp4'))

# --- Pass 2: render kept tracks; HOLD each mask between refinements so it
# stays painted (re-tightening every REFINE_EVERY frames) instead of blinking.
cards = []
last_mask = {}                       # track_id -> most recent SAM 2 mask (bool HxW)
for rec in per_frame:
    img = cv2.imread(f"{TMP}/{rec['k']:05d}.jpg")
    H, W = img.shape[:2]
    present = [d for d in rec['dets'] if d['id'] in keep]
    # refresh the cache from any masks SAM 2 produced on this frame
    for d in present:
        if 'mask' in d:
            last_mask[d['id']] = d['mask']
    # 1) paint held masks first (so boxes stay crisp on top)
    for d in present:
        m = last_mask.get(d['id'])
        if m is not None:
            col_bgr = col_rgb(d['id'])[::-1]
            img[m] = (0.5*img[m] + 0.5*np.array(col_bgr)).astype(np.uint8)
    # 2) draw boxes + labels and build cards
    for d in present:
        col_bgr = col_rgb(d['id'])[::-1]
        x0, y0, x1, y1 = d['bbox']
        cv2.rectangle(img, (x0, y0), (x1, y1), col_bgr, 2)
        cv2.putText(img, f"#{d['id']} {NAMES[d['cls']]} {d['conf']:.2f}",
                    (x0, max(12, y0-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col_bgr, 1, cv2.LINE_AA)
        card = {'frame': rec['k'], 'track_id': d['id'], 'class': NAMES[d['cls']],
                'confidence': round(d['conf'], 4), 'bbox': d['bbox']}
        if 'mask' in d:                 # area + polygons only on true SAM 2 frames
            area = int(d['mask'].sum())
            card['area_px'] = area
            card['coverage_frac'] = round(area/(H*W), 5)
            card['polygons'] = mask_to_polygons(d['mask'], W, H)
        cards.append(card)
    cv2.imwrite(f"{TMP}/{rec['k']:05d}.jpg", img)

# --- encode overlay + write per-frame and per-instance cards ---
local_mp4 = '/content/internal_assets_demo.mp4'
subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
                '-framerate', str(round(fps)), '-i', f'{TMP}/%05d.jpg',
                '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2', '-c:v', 'libx264',
                '-pix_fmt', 'yuv420p', '-movflags', '+faststart', local_mp4], check=True)
shutil.copy(local_mp4, os.path.join(OUT_DIR, 'internal_assets_demo.mp4'))

by_track = defaultdict(list)
for c in cards:
    by_track[c['track_id']].append(c)
track_cards = []
for tid, cs in by_track.items():
    areas = [c['area_px'] for c in cs if 'area_px' in c]
    track_cards.append({'track_id': tid, 'class': cs[0]['class'],
                        'first_frame': cs[0]['frame'], 'last_frame': cs[-1]['frame'],
                        'n_frames': len(cs),
                        'peak_conf': round(max(c['confidence'] for c in cs), 4),
                        'peak_area_px': max(areas) if areas else None})
meta = {
    'schema_version': 1,
    'source_video': os.path.basename(VIDEO_PATH),
    'domain': DOMAIN,
    'asset': ASSET,
    'fps': round(fps, 3),
    'enc_fps': int(round(fps)),          # the fps ffmpeg encoded at; app maps frame = round(t*enc_fps)
    'start_frame': int(start_f),         # provenance (original-video index); cards are clip-local 0-based
    'n_frames': len(per_frame),
    'width': int(W), 'height': int(H),
    'classes': [NAMES[i] for i in range(len(NAMES))],
    'refine_every': REFINE_EVERY,
    'min_track_len': MIN_TRACK_LEN,
    'conf_threshold': CONF,
    'frame_indexing': 'clip_local_0based',
}
assert all(c['frame'] < len(per_frame) for c in cards), 'clip-local frame index out of range'
json.dump(cards, open(os.path.join(OUT_DIR, 'detection_cards.json'), 'w'), indent=1)
json.dump(track_cards, open(os.path.join(OUT_DIR, 'track_cards.json'), 'w'), indent=1)
json.dump(meta, open(os.path.join(OUT_DIR, 'meta.json'), 'w'), indent=1)
shutil.rmtree(TMP, ignore_errors=True)
print(f"{len(keep)} tracked instances, {len(cards)} per-frame dets -> {OUT_DIR}")

In [ ]:
# Inline preview of the overlay (Colab)
if IN_COLAB:
    from IPython.display import HTML
    from base64 import b64encode
    data = b64encode(open('/content/internal_assets_demo.mp4', 'rb').read()).decode()
    display(HTML(f'<video width=720 controls><source src="data:video/mp4;base64,{data}" '
                 f'type="video/mp4"></video>'))

## Where to go from here
If segmentation quality is **good given a prompt**, the remaining problem is *automation* — generating the prompts without a human. Options, roughly in order of effort:
- **Detector → SAM 2:** fine-tune a detector (YOLO/DETR) on labelled defects to emit boxes, feed those boxes as SAM 2 prompts. This is where most of your accuracy will live.
- **Automatic Mask Generator + classifier:** segment everything, then classify each mask. Heavier compute, no labelled boxes needed up front.
- **Tiling** for small defects that the 1024px resize loses.
- **Fine-tune SAM 2 itself** ([training/](../training/)) only if mask *boundaries* on your domain are poor even with good prompts.

If quality is **poor even with good prompts**, tell me what's failing (drift, bleeding into background, missing thin features) and we'll target the fix.